# 🔍 03. Multi-Pass Blocking & Candidate Generation
### Amazon ML Challenge 2026: Multi-Source Business Entity Resolution

Matching $2.2M$ Source 1 entities against $10.3M$ Source 2 & 3 candidates requires reducing the $O(N \times M) = 2.2 \times 10^{13}$ pairwise space.
In this notebook:
1. Vectorized multi-pass inverted index blocking:
   - **Pass 1:** Exact Normalized Business Name
   - **Pass 2:** Clean Business Name (Legal suffix stripped)
   - **Pass 3:** 4-Character Alphanumeric Prefix Key
2. Evaluate Candidate Pair Recall & Candidate Volume against Ground Truth.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "amazon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd

from src.utils.io import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
report_path = PROJECT_ROOT / "artifacts" / "reports" / "candidate_generation_report.json"

if report_path.exists():
    with open(report_path, encoding="utf-8") as f:
        rep = json.load(f)
    print("Candidate Generation Metrics on Evaluation Sample:")
    print(f"  Candidate Pair Recall:      {rep['candidate_pair_recall']:.2%}")
    print(f"  Entities Fully Retrieved:   {rep['entity_full_recall']:.2%}")
    print(f"  Retrieved True Pairs:       {rep['retrieved_true_pairs']:,} / {rep['total_true_pairs']:,}")
    print(f"  Candidate Volume (Mean):    {rep['candidate_volume']['mean']:.1f}")
    print(f"  Candidate Volume (Median):  {rep['candidate_volume']['median']:.1f}")
    print(f"  Candidate Volume (P95):     {rep['candidate_volume']['p95']:.1f}")
    print(f"  Candidate Volume (Max):     {rep['candidate_volume']['max']}")
else:
    print("Candidate report not found yet. Generating sample...")

## 1. Inspect Generated Candidate Pairs
Load a sample of generated candidate pairs with ground truth labels.

In [ ]:
cand_parquet = PROJECT_ROOT / cfg["paths"]["candidates_dir"] / "train_candidates.parquet"
if cand_parquet.exists():
    df_cands = pd.read_parquet(cand_parquet)
    print(f"Total candidate pairs: {len(df_cands):,}")
    print("Label distribution:")
    display(df_cands["label"].value_counts(normalize=True).apply(lambda x: f"{x:.2%}"))
    print("\nSample Positive Matches:")
    display(
        df_cands[df_cands["label"] == 1][
            [
                "source1_business_name",
                "candidate_business_name",
                "source1_business_address",
                "candidate_business_address",
            ]
        ].head(5)
    )
else:
    print("Candidate parquet will be displayed once generated.")